In [26]:
"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║               TOOL-RAG  /  TOOLFORMER-RAG  ENGINE                             ║
║               tool_rag_engine.py  — Core Pipeline                             ║
║                                                                                ║
║  Inspired by:                                                                  ║
║    • Toolformer (Schick et al., 2023) — LM learns to call APIs inline         ║
║    • FLARE (Jiang et al., 2023)       — Active retrieval during generation     ║
║    • ReAct  (Yao et al., 2023)        — Reasoning + Acting interleaved         ║
║    • ToolkenGPT (Hao et al., 2023)   — Tool tokens embedded in LM vocab       ║
║                                                                                ║
║  LLM         : Azure OpenAI (GPT-4 / GPT-35-turbo)                            ║
║  Embeddings  : HuggingFace sentence-transformers (local, zero API cost)        ║
║  Vector DB   : ChromaDB (persistent HNSW index)                               ║
║  Framework   : LangChain (LCEL chains + custom tool dispatch)                 ║
╚══════════════════════════════════════════════════════════════════════════════════╝

WHAT MAKES THIS "TOOL-RAG" (NOT STANDARD RAG)
══════════════════════════════════════════════
Standard RAG  : retrieve-once → generate (fixed pipeline, no agency)
Agentic RAG   : LLM decides IF to retrieve (one loop before generation)
Tool-RAG      : LLM calls tools INLINE during generation, interleaving
                reasoning ↔ tool_calls ↔ observations in a single pass.
                Tools are first-class citizens: Calculator, KnowledgeBase,
                DateResolver, FactChecker, Summarizer, Comparator.
                The model generates "[TOOL: name(args)]" markers mid-text,
                the engine intercepts, executes, substitutes the result,
                and the model CONTINUES generating with real grounded context.

TOOLFORMER-STYLE INLINE TOOL CALLS
════════════════════════════════════
Generation step produces tokens like:
  "The Transformer has [TOOL:KB(multi-head attention)] and was trained
   for [TOOL:CALC(12 * 64)] = 768 hidden dims."

Engine intercepts → executes → substitutes → model resumes:
  "The Transformer has [Multi-Head Attention allows joint
   attention across 8 subspaces...] and was trained for 768 hidden dims."

TOOLS AVAILABLE
═══════════════
  1. KB         — Knowledge Base retrieval (ChromaDB semantic search)
  2. CALC       — Safe arithmetic evaluator
  3. DATE       — Date arithmetic / formatting
  4. COMPARE    — Side-by-side comparison via retrieved context
  5. SUMMARIZE  — Targeted summarization of retrieved chunks
  6. FACTCHECK  — Verify a claim against the knowledge base
  7. DEFINE     — Retrieve and format a definition/concept
"""

'\n╔══════════════════════════════════════════════════════════════════════════════════╗\n║               TOOL-RAG  /  TOOLFORMER-RAG  ENGINE                             ║\n║               tool_rag_engine.py  — Core Pipeline                             ║\n║                                                                                ║\n║  Inspired by:                                                                  ║\n║    • Toolformer (Schick et al., 2023) — LM learns to call APIs inline         ║\n║    • FLARE (Jiang et al., 2023)       — Active retrieval during generation     ║\n║    • ReAct  (Yao et al., 2023)        — Reasoning + Acting interleaved         ║\n║    • ToolkenGPT (Hao et al., 2023)   — Tool tokens embedded in LM vocab       ║\n║                                                                                ║\n║  LLM         : Azure OpenAI (GPT-4 / GPT-35-turbo)                            ║\n║  Embeddings  : HuggingFace sentence-transformers (local, zero API cost)   

In [27]:

# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────
from __future__ import annotations

import os
import re
import ast
import sys
import json
import math
import time
import hashlib
import textwrap
import warnings
import operator
import datetime
import traceback
from enum import Enum
from pathlib import Path
from typing import (
    Any, Callable, Dict, List, Optional,
    Tuple, Union, NamedTuple
)
import threading
import queue
import uuid 
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")


In [28]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [29]:

import chromadb
from chromadb.config import Settings

In [30]:


# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Ollama (local LLM) ─────────────────────────────────────────────
    OLLAMA_BASE_URL: str = field(default_factory=lambda: os.getenv(
        "OLLAMA_BASE_URL", "http://localhost:11434"))
    OLLAMA_MODEL: str    = field(default_factory=lambda: os.getenv(
        "OLLAMA_MODEL", "qwen2.5-coder:7b"))

    # ── Embedding (HuggingFace local) ─────────────────────────────────
    EMBEDDING_MODEL: str          = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str         = field(default_factory=lambda: os.getenv(
        "EMBEDDING_DEVICE", "cpu"))

    # ── ChromaDB ─────────────────────────────────────────────────────
    CHROMA_BASE_DIR: str          = "./chroma_multi_agent"
    # Each domain agent gets its own collection
    DOMAIN_COLLECTIONS: Dict[str, str] = field(default_factory=lambda: {
        "nlp_research":   "ma_nlp_research",
        "systems_infra":  "ma_systems_infra",
        "math_reasoning": "ma_math_reasoning",
        "general":        "ma_general",
    })

    # ── Retrieval ─────────────────────────────────────────────────────
    TOP_K: int           = 4
    FETCH_K: int         = 8
    MMR_LAMBDA: float    = 0.65
    CHUNK_SIZE: int      = 750
    CHUNK_OVERLAP: int   = 130

    # ── LLM ──────────────────────────────────────────────────────────
    LLM_TEMPERATURE: float = 0.0
    LLM_MAX_TOKENS: int    = 1024

    # ── Multi-Agent ───────────────────────────────────────────────────
    MAX_AGENT_RETRIES: int    = 2
    AGENT_TIMEOUT_SEC: float  = 30.0
    ENABLE_PARALLEL: bool     = True   # run domain agents in parallel

    def is_configured(self) -> bool:
        """Check Ollama is reachable."""
        import urllib.request
        try:
            urllib.request.urlopen(self.OLLAMA_BASE_URL, timeout=3)
            return True
        except Exception:
            return False

In [31]:
# ══════════════════════════════════════════════════════════════════════
# TERMINAL LOGGER
# ══════════════════════════════════════════════════════════════════════

class C:
    RESET   = "\033[0m"; BOLD    = "\033[1m"; DIM     = "\033[2m"
    CYAN    = "\033[96m"; GREEN  = "\033[92m"; YELLOW  = "\033[93m"
    RED     = "\033[91m"; MAGENTA= "\033[95m"; BLUE    = "\033[94m"
    WHITE   = "\033[97m"; ORANGE = "\033[33m"
    # Agent-specific colours
    ORCH    = "\033[94m"   # blue   — Orchestrator
    RET     = "\033[96m"   # cyan   — Retriever
    NLP     = "\033[92m"   # green  — NLP domain agent
    SYS     = "\033[33m"   # orange — Systems domain agent
    MATH    = "\033[95m"   # magenta— Math domain agent
    CRIT    = "\033[91m"   # red    — Critic agent
    SYNTH   = "\033[93m"   # yellow — Synthesizer agent


AGENT_COLOURS: Dict[str, str] = {
    "Orchestrator":       C.ORCH,
    "RetrieverAgent":     C.RET,
    "NLPResearchAgent":   C.NLP,
    "SystemsAgent":       C.SYS,
    "MathReasoningAgent": C.MATH,
    "CriticAgent":        C.CRIT,
    "SynthesizerAgent":   C.SYNTH,
}

W = 76  # terminal width


class Log:
    _lock = threading.Lock()

    @staticmethod
    def _p(*args):
        with Log._lock:
            print(*args)

    @staticmethod
    def banner(title: str):
        Log._p(f"\n{C.BOLD}{C.WHITE}{'═'*W}")
        Log._p(f"  {title}")
        Log._p(f"{'═'*W}{C.RESET}")

    @staticmethod
    def section(title: str):
        Log._p(f"\n{C.BOLD}{C.CYAN}{'━'*W}")
        Log._p(f"  {title}")
        Log._p(f"{'━'*W}{C.RESET}")

    @staticmethod
    def agent(agent_name: str, msg: str):
        colour = AGENT_COLOURS.get(agent_name, C.WHITE)
        tag = f"[{agent_name}]"
        Log._p(f"{C.BOLD}{colour}{tag:22s}{C.RESET} {msg}")

    @staticmethod
    def step(msg: str):
        Log._p(f"\n{C.BOLD}{C.BLUE}▶ {msg}{C.RESET}")

    @staticmethod
    def ok(msg: str):
        Log._p(f"{C.GREEN}  ✓ {msg}{C.RESET}")

    @staticmethod
    def warn(msg: str):
        Log._p(f"{C.YELLOW}  ⚠  {msg}{C.RESET}")

    @staticmethod
    def err(msg: str):
        Log._p(f"{C.RED}  ✗ {msg}{C.RESET}")

    @staticmethod
    def info(msg: str):
        Log._p(f"{C.DIM}  · {msg}{C.RESET}")

    @staticmethod
    def msg_send(sender: str, receiver: str, msg_type: str):
        sc = AGENT_COLOURS.get(sender, C.WHITE)
        rc = AGENT_COLOURS.get(receiver, C.WHITE)
        Log._p(f"{C.DIM}  📨 {sc}{sender}{C.RESET}{C.DIM} → "
               f"{rc}{receiver}{C.RESET}{C.DIM} [{msg_type}]{C.RESET}")

    @staticmethod
    def answer_box(query: str, answer: str, meta: Dict[str, Any]):
        Log._p(f"\n{C.BOLD}{C.GREEN}{'═'*W}")
        Log._p(f"  FINAL MULTI-AGENT ANSWER")
        Log._p(f"{'═'*W}{C.RESET}")
        Log._p(f"{C.BOLD}  Q: {query}{C.RESET}\n")
        for line in textwrap.wrap(answer, W - 4):
            Log._p(f"  {line}")
        Log._p(f"\n{C.DIM}{'─'*W}")
        agents = meta.get("agents_used", [])
        Log._p(f"  Agents activated : {agents}")
        Log._p(f"  Docs retrieved   : {meta.get('total_docs', 0)}")
        Log._p(f"  Messages passed  : {meta.get('message_count', 0)}")
        Log._p(f"  Critique score   : {meta.get('critique_score', 'N/A')}/10")
        Log._p(f"  Elapsed          : {meta.get('elapsed', 0):.2f}s")
        Log._p(f"{'─'*W}{C.RESET}")



In [32]:

# ══════════════════════════════════════════════════════════════════════
# TYPED MESSAGE SYSTEM
# ══════════════════════════════════════════════════════════════════════

class MsgType(str, Enum):
    # Orchestrator → domain agents
    TASK_ASSIGN       = "TASK_ASSIGN"
    # Domain agents → Orchestrator / Synthesizer
    PARTIAL_ANSWER    = "PARTIAL_ANSWER"
    RETRIEVAL_RESULT  = "RETRIEVAL_RESULT"
    # Orchestrator → Critic
    CRITIQUE_REQUEST  = "CRITIQUE_REQUEST"
    # Critic → Synthesizer
    CRITIQUE_RESULT   = "CRITIQUE_RESULT"
    # Orchestrator → Synthesizer
    SYNTHESIZE_REQUEST = "SYNTHESIZE_REQUEST"
    # Synthesizer → Orchestrator
    FINAL_ANSWER      = "FINAL_ANSWER"
    # Any agent
    ERROR             = "ERROR"
    STATUS            = "STATUS"
    HEARTBEAT         = "HEARTBEAT"


@dataclass
class Message:
    """Typed inter-agent message."""
    msg_id:    str       = field(default_factory=lambda: str(uuid.uuid4())[:8])
    sender:    str       = ""
    receiver:  str       = ""
    msg_type:  MsgType   = MsgType.STATUS
    payload:   Dict[str, Any] = field(default_factory=dict)
    timestamp: float     = field(default_factory=time.time)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "msg_id":    self.msg_id,
            "sender":    self.sender,
            "receiver":  self.receiver,
            "msg_type":  self.msg_type.value,
            "payload":   self.payload,
            "timestamp": self.timestamp,
        }



In [33]:

# ══════════════════════════════════════════════════════════════════════
# MESSAGE BUS
# ══════════════════════════════════════════════════════════════════════

class MessageBus:
    """
    Thread-safe publish/subscribe message bus.
    Agents register inboxes; senders push messages;
    receivers pull from their own queue.
    """

    def __init__(self):
        self._inboxes: Dict[str, queue.Queue] = {}
        self._history: List[Message] = []
        self._lock = threading.Lock()

    def register(self, agent_name: str):
        with self._lock:
            if agent_name not in self._inboxes:
                self._inboxes[agent_name] = queue.Queue()

    def send(self, msg: Message):
        with self._lock:
            self._history.append(msg)
        Log.msg_send(msg.sender, msg.receiver, msg.msg_type.value)
        inbox = self._inboxes.get(msg.receiver)
        if inbox is not None:
            inbox.put(msg)
        else:
            Log.warn(f"Bus: no inbox for '{msg.receiver}'")

    def receive(
        self,
        agent_name: str,
        timeout: float = 30.0,
    ) -> Optional[Message]:
        inbox = self._inboxes.get(agent_name)
        if inbox is None:
            return None
        try:
            return inbox.get(timeout=timeout)
        except queue.Empty:
            return None

    def receive_all_waiting(self, agent_name: str) -> List[Message]:
        """Drain all currently queued messages for an agent."""
        inbox = self._inboxes.get(agent_name)
        msgs: List[Message] = []
        if inbox is None:
            return msgs
        while True:
            try:
                msgs.append(inbox.get_nowait())
            except queue.Empty:
                break
        return msgs

    def broadcast(self, msg: Message, receivers: List[str]):
        for r in receivers:
            m = Message(
                sender=msg.sender, receiver=r,
                msg_type=msg.msg_type, payload=msg.payload,
            )
            self.send(m)

    @property
    def message_count(self) -> int:
        return len(self._history)

    def history_summary(self) -> List[str]:
        return [
            f"{m.sender}→{m.receiver} [{m.msg_type.value}]"
            for m in self._history
        ]


In [34]:

# ══════════════════════════════════════════════════════════════════════
# AGENT RESULT TYPE
# ══════════════════════════════════════════════════════════════════════

@dataclass
class AgentResult:
    agent_name:  str
    domain:      str
    answer:      str
    sources:     List[str]
    doc_count:   int
    confidence:  float          # 0.0–1.0
    elapsed_sec: float
    error:       Optional[str]  = None

    @property
    def success(self) -> bool:
        return self.error is None and bool(self.answer)


In [35]:
"""

Demo corpus of 9 documents partitioned across three domain collections:

  DOMAIN 1 — NLP Research      (papers: Attention, BERT, GPT-3, RAG)
  DOMAIN 2 — Systems / Infra   (ChromaDB, FAISS, LangChain, AutoGen)
  DOMAIN 3 — Math / Reasoning  (CoT, MathRAG, ReAct, FLARE)

Each domain gets its own ChromaDB collection. The Retriever Agent
queries all three and returns ranked results per domain.

📄 Reference PDFs (all publicly available):
  • Attention Is All You Need     https://arxiv.org/pdf/1706.03762
  • BERT                          https://arxiv.org/pdf/1810.04805
  • GPT-3                         https://arxiv.org/pdf/2005.14165
  • RAG paper                     https://arxiv.org/pdf/2005.11401
  • ReAct                         https://arxiv.org/pdf/2210.03629
  • AutoGen                       https://arxiv.org/pdf/2308.08155
  • Chain-of-Thought Prompting    https://arxiv.org/pdf/2201.11903
  • CAMEL                         https://arxiv.org/pdf/2303.17760
  • Self-RAG                      https://arxiv.org/pdf/2310.11511
"""


'\n\nDemo corpus of 9 documents partitioned across three domain collections:\n\n  DOMAIN 1 — NLP Research      (papers: Attention, BERT, GPT-3, RAG)\n  DOMAIN 2 — Systems / Infra   (ChromaDB, FAISS, LangChain, AutoGen)\n  DOMAIN 3 — Math / Reasoning  (CoT, MathRAG, ReAct, FLARE)\n\nEach domain gets its own ChromaDB collection. The Retriever Agent\nqueries all three and returns ranked results per domain.\n\n📄 Reference PDFs (all publicly available):\n  • Attention Is All You Need     https://arxiv.org/pdf/1706.03762\n  • BERT                          https://arxiv.org/pdf/1810.04805\n  • GPT-3                         https://arxiv.org/pdf/2005.14165\n  • RAG paper                     https://arxiv.org/pdf/2005.11401\n  • ReAct                         https://arxiv.org/pdf/2210.03629\n  • AutoGen                       https://arxiv.org/pdf/2308.08155\n  • Chain-of-Thought Prompting    https://arxiv.org/pdf/2201.11903\n  • CAMEL                         https://arxiv.org/pdf/2303.17760\n  

In [36]:

# ─────────────────────────────────────────────────────────────────────
# DOMAIN 1 — NLP RESEARCH
# ─────────────────────────────────────────────────────────────────────
NLP_RESEARCH_DOCS: List[Dict[str, str]] = [
    {
        "id": "transformer_arch",
        "source": "Attention Is All You Need — Vaswani et al. (2017)",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer model relies entirely on self-attention to draw global dependencies between
input and output, dispensing with recurrence and convolutions. The encoder and decoder each
consist of N=6 stacked identical layers.

Scaled Dot-Product Attention:  Attention(Q,K,V) = softmax(Q·K^T / √d_k) · V
Multi-Head Attention: MHA(Q,K,V) = Concat(head_1,...,head_h)·W^O
  where head_i = Attention(Q·W_i^Q, K·W_i^K, V·W_i^V)

Base model: d_model=512, h=8 heads, d_k=d_v=64, d_ff=2048.  Parameters: 65M.
Large model: d_model=1024, h=16, d_ff=4096.  Parameters: 213M.
Position encodings: PE(pos,2i) = sin(pos/10000^(2i/d_model)).
Results: EN→DE 28.4 BLEU (prior SOTA 26.0), EN→FR 41.0 BLEU.
Training: Adam, β1=0.9, β2=0.98, ε=1e-9, warmup_steps=4000.
"""
    },
    {
        "id": "bert",
        "source": "BERT: Pre-training of Deep Bidirectional Transformers — Devlin et al. (2018)",
        "url": "https://arxiv.org/abs/1810.04805",
        "content": """
BERT (Bidirectional Encoder Representations from Transformers) pre-trains deep bidirectional
representations from unlabeled text by jointly conditioning on both left and right context
at every layer. Two pre-training tasks: Masked Language Model (MLM) and Next Sentence
Prediction (NSP).

MLM: randomly mask 15% of tokens; predict them using the full bidirectional context.
NSP: train on sentence pairs; predict if sentence B follows sentence A.

BERT_BASE: 12 layers, 768 hidden, 12 heads, 110M params.
BERT_LARGE: 24 layers, 1024 hidden, 16 heads, 340M params.

Fine-tuning: prepend [CLS] token; use its representation for classification.
GLUE benchmark: BERT_LARGE = 80.5% (prior SOTA 72.8%).
SQuAD v1.1 F1 = 93.2% (human performance 91.2%).
SQuAD v2.0 F1 = 83.1%. SWAG: 86.3% (prior SOTA 59.2%).
Pre-training: BooksCorpus (800M words) + English Wikipedia (2,500M words).
"""
    },
    {
        "id": "rag_paper",
        "source": "Retrieval-Augmented Generation for Knowledge-Intensive NLP — Lewis et al. (2020)",
        "url": "https://arxiv.org/abs/2005.11401",
        "content": """
RAG combines a pre-trained seq2seq model (generator) with a non-parametric dense vector
retrieval component (retriever). The retriever uses Dense Passage Retrieval (DPR) with
dual BERT encoders over a FAISS index of 21M Wikipedia 100-word passages.

RAG-Sequence: p(y|x) = Σ_z p_η(z|x) · p_θ(y|x,z)
RAG-Token:    p(y|x) = Π_i Σ_z p_η(z|x) · p_θ(y_i|x,z,y_{1:i-1})

The non-parametric memory (document index) can be updated without retraining the generator.
Results: Natural Questions 44.5 EM vs T5 42.1; WebQA 45.5 vs T5 37.4; TriviaQA 56.1.
CuratedTrec: RAG 52.0 vs BERT 33.0. Open MS-MARCO NLG: RAG sets new state of the art.
Generator: BART-large (400M params). DPR retriever: two BERT-base encoders.
"""
    },
    {
        "id": "self_rag",
        "source": "Self-RAG: Learning to Retrieve, Generate and Critique — Asai et al. (2023)",
        "url": "https://arxiv.org/abs/2310.11511",
        "content": """
Self-RAG trains a single LM that adaptively retrieves passages on demand and uses
reflection tokens to generate and reflect upon its own outputs. The model learns four
types of reflection tokens:

  [Retrieve]       — should the model retrieve passages for this span?
  [IsREL]          — is the retrieved passage relevant?
  [IsSUP]          — is the output supported by the passage?
  [IsUSE]          — is the response useful to the user?

Unlike standard RAG (always retrieve) or no-retrieval (never), Self-RAG retrieves only
when needed. During inference the model generates [Retrieve=Yes/No] tokens to gate
retrieval. The model is fine-tuned end-to-end with both the LM objective and reflection
token prediction.

Results: Self-RAG (7B) outperforms ChatGPT on PopQA, TriviaQA, ASQA, FactScore,
and ARC-Challenge. On ASQA EM-recall: Self-RAG 38.2 vs RAG 35.9 vs ChatGPT 35.1.
On FactScore (biography factuality): 72.4 vs ChatGPT 60.1.
"""
    },
]


# ─────────────────────────────────────────────────────────────────────
# DOMAIN 2 — SYSTEMS / INFRASTRUCTURE
# ─────────────────────────────────────────────────────────────────────
SYSTEMS_INFRA_DOCS: List[Dict[str, str]] = [
    {
        "id": "chromadb_sys",
        "source": "ChromaDB: Open-Source AI Application Database",
        "url": "https://docs.trychroma.com/",
        "content": """
ChromaDB is an open-source AI-native vector database built for embedding storage and
semantic similarity search. It uses HNSW (Hierarchical Navigable Small World) for
approximate nearest neighbor search with O(log n) query and insert complexity.

Storage backends: in-memory (ephemeral) and persistent (SQLite + hnswlib on disk).
Distance metrics: cosine similarity, L2 Euclidean, inner product.
Metadata filtering: WHERE clause combined with vector similarity.

LangChain integration:
  chroma = Chroma.from_documents(docs, embeddings, persist_directory="./chroma")
  retriever = chroma.as_retriever(search_type="mmr",
              search_kwargs={"k": 4, "fetch_k": 8, "lambda_mult": 0.65})

MMR (Maximal Marginal Relevance): balances relevance and diversity.
  MMR = argmax_d [λ·sim(d,q) − (1−λ)·max_{d'∈S} sim(d,d')]
HNSW graph: each node connects to M nearest neighbors during construction.
Multi-collection: each logical namespace (e.g. domain) gets its own collection.
Client API: PersistentClient, EphemeralClient, HttpClient (for server mode).
"""
    },
    {
        "id": "autogen",
        "source": "AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation — Wu et al. (2023)",
        "url": "https://arxiv.org/abs/2308.08155",
        "content": """
AutoGen is a framework that enables next-generation LLM applications via multi-agent
conversations. Agents in AutoGen are conversable (can send/receive messages), customizable
(LLM-backed, tool-backed, or human-in-the-loop), and composable into flexible topologies.

Core agent types:
  AssistantAgent    — backed by LLM (e.g. GPT-4), processes requests, generates replies
  UserProxyAgent    — represents human input, can execute code, call tools
  GroupChatManager  — orchestrates multi-agent group conversations

Key design patterns:
  1. Two-agent chat (User ↔ Assistant) — simplest collaborative unit
  2. Group chat — N agents discuss, manager routes turns
  3. Hierarchical — outer orchestrator delegates to inner multi-agent groups
  4. Sequential pipeline — output of agent_i feeds agent_{i+1}

AutoGen vs LangChain agents:
  AutoGen excels at conversational coordination; LangChain at pipeline chaining.
  AutoGen supports code execution in sandboxed environments natively.

Results: AutoGen-based systems solve 69% of HumanEval coding tasks vs 55% for GPT-4 alone.
On MATH benchmark: multi-agent AutoGen achieves 72.3% vs single-agent 56.8%.
"""
    },
    {
        "id": "langchain_sys",
        "source": "LangChain: Building LLM Applications with Composable Components",
        "url": "https://python.langchain.com/docs/introduction",
        "content": """
LangChain is an open-source framework for building LLM-powered applications. Its core
abstractions enable composable, modular pipelines via LCEL (LangChain Expression Language).

LCEL uses the | pipe operator for chaining Runnables:
  chain = prompt | llm | StrOutputParser()
  rag_chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm

Key agent types:
  create_react_agent()         — Thought/Action/Observation loop (ReAct)
  create_openai_tools_agent()  — native OpenAI tool/function calling
  AgentExecutor               — wrapper with retry, memory, intermediate steps

Multi-agent in LangChain:
  Use multiple AgentExecutor instances, one per specialized agent.
  Pass context between agents via shared memory or structured messages.
  LangGraph (built on LangChain) adds explicit state machine control for multi-agent.

Retrieval chain:
  retriever → format_docs → prompt → llm → output_parser
  Supports: ChromaDB, FAISS, Pinecone, Weaviate, Qdrant, pgvector, 30+ others.
"""
    },
]


# ─────────────────────────────────────────────────────────────────────
# DOMAIN 3 — MATH / REASONING
# ─────────────────────────────────────────────────────────────────────
MATH_REASONING_DOCS: List[Dict[str, str]] = [
    {
        "id": "chain_of_thought",
        "source": "Chain-of-Thought Prompting Elicits Reasoning in LLMs — Wei et al. (2022)",
        "url": "https://arxiv.org/abs/2201.11903",
        "content": """
Chain-of-thought (CoT) prompting generates a series of intermediate reasoning steps before
arriving at the final answer. This technique substantially improves LLM performance on
arithmetic, commonsense, and symbolic reasoning tasks.

Standard prompting: Question → Answer
CoT prompting:      Question → Reasoning chain → Answer

Example (GSM8K):
  Q: If there are 3 cars in the parking lot and 2 more arrive, how many total?
  A: There are 3 cars. 2 more arrive. 3 + 2 = 5. The answer is 5.

Results on GSM8K: PaLM (540B) CoT = 56.9% vs standard = 17.9%.
Results on MATH: GPT-4 CoT = 52.9% vs standard = 42.2%.
Results on StrategyQA: PaLM CoT = 75.4% vs standard = 65.3%.

CoT is an emergent property: only appears in models ≥ ~100B parameters.
Zero-shot CoT ("Let's think step by step"): Kojima et al. 2022, works surprisingly well.
Self-consistency (Wang et al. 2022): sample N CoT paths, majority vote → +10-20% accuracy.
"""
    },
    {
        "id": "react_reasoning",
        "source": "ReAct: Synergizing Reasoning and Acting in Language Models — Yao et al. (2023)",
        "url": "https://arxiv.org/abs/2210.03629",
        "content": """
ReAct interleaves chain-of-thought reasoning with external action calls in a unified loop:
  Thought → Action → Observation → Thought → ... → Final Answer

Unlike pure CoT (no external grounding) or act-only (no reasoning), ReAct combines both.
This allows the model to: maintain state, handle errors, refine strategy mid-task.

Reasoning trace example:
  Thought: I need to find the capital of the country where the Eiffel Tower is.
  Action: Search[Eiffel Tower location]
  Observation: The Eiffel Tower is in Paris, France.
  Thought: France's capital is Paris, which is also where the tower is.
  Final Answer: Paris

HotpotQA (multi-hop QA): ReAct 50.3 EM vs CoT 29.4 vs Act-only 28.7.
ALFWorld (household tasks): ReAct 71% vs Act-only 45%.
WebShop (e-commerce): ReAct 40.0% success vs SOTA IL+RL 28.7%.
Hallucination reduction: ReAct grounds every factual claim in an Observation.
"""
    },
    {
        "id": "flare_active",
        "source": "FLARE: Active Retrieval Augmented Generation — Jiang et al. (2023)",
        "url": "https://arxiv.org/abs/2305.06983",
        "content": """
FLARE (Forward-Looking Active REtrieval) monitors generation confidence token-by-token.
When a generated token has probability < δ (typically 0.4), that span is flagged as
low-confidence. The system uses the tentative tokens as a retrieval query, fetches
documents, and regenerates the sentence with the retrieved context.

Algorithm:
  1. Generate sentence s with current context C
  2. For each token t_i in s: if p(t_i | C, t_{1:i-1}) < δ → flag uncertain span
  3. Form query q from non-masked tokens in s
  4. Retrieve docs D = Retrieve(q)
  5. Regenerate s using C + D
  6. Append s to C, continue to next sentence

FLARE vs Standard RAG:
  Standard RAG retrieves once at the start — misses needs that arise mid-generation.
  FLARE retrieves exactly when and where needed.

Results: StrategyQA 60.1 vs standard RAG 57.8.
  ASQA EM-recall 43.1 vs 39.7 (standard RAG). 2WikiMultihop 46.2 vs 37.9.
Adaptive retrieval rate: FLARE retrieves for ~20-40% of sentences depending on task.
"""
    },
]


# ─────────────────────────────────────────────────────────────────────
# DOMAIN MAP  — used by DocumentProcessor
# ─────────────────────────────────────────────────────────────────────
DOMAIN_CORPUS: Dict[str, List[Dict[str, str]]] = {
    "nlp_research":   NLP_RESEARCH_DOCS,
    "systems_infra":  SYSTEMS_INFRA_DOCS,
    "math_reasoning": MATH_REASONING_DOCS,
}

# Primary PDF to download (Attention Is All You Need — most universally cited)
PRIMARY_PDF = {
    "url":   "https://arxiv.org/pdf/1706.03762",
    "local": "./attention_is_all_you_need.pdf",
    "domain": "nlp_research",
    "source": "Attention Is All You Need — Vaswani et al. (2017)",
    "url_ref": "https://arxiv.org/abs/1706.03762",
}

# All reference URLs for display
ALL_REFERENCES = [
    ("Attention Is All You Need",        "https://arxiv.org/abs/1706.03762"),
    ("BERT",                             "https://arxiv.org/abs/1810.04805"),
    ("RAG Paper",                        "https://arxiv.org/abs/2005.11401"),
    ("Self-RAG",                         "https://arxiv.org/abs/2310.11511"),
    ("AutoGen",                          "https://arxiv.org/abs/2308.08155"),
    ("Chain-of-Thought Prompting",       "https://arxiv.org/abs/2201.11903"),
    ("ReAct",                            "https://arxiv.org/abs/2210.03629"),
    ("FLARE",                            "https://arxiv.org/abs/2305.06983"),
    ("ChromaDB Docs",                    "https://docs.trychroma.com/"),
    ("LangChain Docs",                   "https://python.langchain.com/docs/introduction"),
]


In [37]:

# ══════════════════════════════════════════════════════════════════════
# DOCUMENT PROCESSOR
# ══════════════════════════════════════════════════════════════════════

class DocumentProcessor:
    """
    Converts corpus entries into chunked, deduped LangChain Documents,
    partitioned by domain.
    """

    def __init__(self, cfg: Config):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg.CHUNK_SIZE,
            chunk_overlap=cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=len,
        )

    def _try_pdf(self, url: str, local_path: str, domain: str,
                 source: str, url_ref: str) -> List[Document]:
        """Attempt to download and parse a PDF into Documents."""
        try:
            import urllib.request
            if not Path(local_path).exists():
                Log.info(f"Downloading PDF: {url}")
                urllib.request.urlretrieve(url, local_path)
                Log.ok(f"PDF saved → {local_path}")
            from langchain_community.document_loaders import PyPDFLoader
            pages = PyPDFLoader(local_path).load()
            for p in pages:
                p.metadata.update({
                    "source": source,
                    "url": url_ref,
                    "domain": domain,
                    "doc_id": "pdf_download",
                })
            Log.ok(f"PDF loaded: {len(pages)} pages → domain '{domain}'")
            return pages
        except Exception as exc:
            Log.warn(f"PDF unavailable ({exc}), using inline corpus only")
            return []

    def build_domain_chunks(self) -> Dict[str, List[Document]]:
        """
        Returns a dict mapping domain_key → list of chunked Documents.
        Optionally enriched with downloaded PDF chunks.
        """
        domain_chunks: Dict[str, List[Document]] = {}

        # Optional PDF enrichment
        pdf_docs = self._try_pdf(
            url=PRIMARY_PDF["url"],
            local_path=PRIMARY_PDF["local"],
            domain=PRIMARY_PDF["domain"],
            source=PRIMARY_PDF["source"],
            url_ref=PRIMARY_PDF["url_ref"],
        )

        for domain_key, entries in DOMAIN_CORPUS.items():
            raw_docs: List[Document] = []

            # Inline corpus documents
            for entry in entries:
                doc = Document(
                    page_content=entry["content"].strip(),
                    metadata={
                        "source": entry["source"],
                        "url":    entry["url"],
                        "domain": domain_key,
                        "doc_id": entry["id"],
                    },
                )
                raw_docs.append(doc)

            # Inject PDF docs into nlp_research domain
            if domain_key == "nlp_research" and pdf_docs:
                raw_docs.extend(pdf_docs)

            # Chunk
            chunks = self.splitter.split_documents(raw_docs)

            # Deduplicate by content hash
            seen: set[str] = set()
            unique: List[Document] = []
            for i, chunk in enumerate(chunks):
                h = hashlib.md5(chunk.page_content.encode()).hexdigest()[:10]
                if h not in seen:
                    seen.add(h)
                    chunk.metadata["chunk_id"]   = i
                    chunk.metadata["chunk_hash"] = h
                    chunk.metadata["char_len"]   = len(chunk.page_content)
                    unique.append(chunk)

            domain_chunks[domain_key] = unique
            Log.ok(f"Domain '{domain_key}': {len(unique)} unique chunks")

        return domain_chunks


In [38]:

# ══════════════════════════════════════════════════════════════════════
# DOMAIN VECTOR STORE
# ══════════════════════════════════════════════════════════════════════

class DomainVectorStore:
    """
    A single ChromaDB collection for one domain.
    Exposes: mmr_retriever(), similarity_search(), get_context_str().
    """

    def __init__(
        self,
        domain_key: str,
        collection_name: str,
        embeddings: HuggingFaceEmbeddings,
        cfg: Config,
    ):
        self.domain_key      = domain_key
        self.collection_name = collection_name
        self.embeddings      = embeddings
        self.cfg             = cfg
        self._store: Optional[Chroma] = None

    def build(self, chunks: List[Document], client: chromadb.PersistentClient) -> "DomainVectorStore":
        """Index chunks into this domain's ChromaDB collection."""
        # Wipe existing collection for a fresh build
        try:
            client.delete_collection(self.collection_name)
        except Exception:
            pass

        self._store = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            client=client,
            collection_name=self.collection_name,
            collection_metadata={"hnsw:space": "cosine"},
        )
        n = self._store._collection.count()
        Log.ok(f"  Collection '{self.collection_name}': {n} vectors indexed")
        return self

    def mmr_retriever(self):
        """Return MMR retriever for this domain."""
        return self._store.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k":           self.cfg.TOP_K,
                "fetch_k":     self.cfg.FETCH_K,
                "lambda_mult": self.cfg.MMR_LAMBDA,
            },
        )

    def similarity_search(
        self, query: str, k: Optional[int] = None
    ) -> List[Tuple[Document, float]]:
        """Cosine similarity search returning (doc, score) pairs."""
        k = k or self.cfg.TOP_K
        return self._store.similarity_search_with_relevance_scores(query, k=k)

    def get_context_str(self, query: str, k: Optional[int] = None) -> str:
        """Retrieve and format top-k results as a numbered context string."""
        results = self.similarity_search(query, k=k)
        parts = []
        for i, (doc, score) in enumerate(results, 1):
            src = doc.metadata.get("source", "Unknown")
            url = doc.metadata.get("url", "")
            parts.append(
                f"[{i}] Source: {src}\n"
                f"    URL: {url}\n"
                f"    Relevance: {score:.3f}\n"
                f"    {doc.page_content.strip()}"
            )
        return "\n\n".join(parts) if parts else "(no results)"

    @property
    def doc_count(self) -> int:
        return self._store._collection.count() if self._store else 0


In [39]:

# ══════════════════════════════════════════════════════════════════════
# MULTI-DOMAIN VECTOR STORE MANAGER
# ══════════════════════════════════════════════════════════════════════

class MultiDomainVectorStore:
    """
    Top-level manager that holds one DomainVectorStore per domain.
    Provides cross-domain search and per-domain access.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.embeddings: Optional[HuggingFaceEmbeddings] = None
        self._domains: Dict[str, DomainVectorStore] = {}
        self._client: Optional[chromadb.PersistentClient] = None

    def init_embeddings(self) -> "MultiDomainVectorStore":
        Log.step("Initialising HuggingFace Embeddings (local, no API key)")
        Log.info(f"Model : {self.cfg.EMBEDDING_MODEL}")
        Log.info(f"Device: {self.cfg.EMBEDDING_DEVICE}")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.cfg.EMBEDDING_MODEL,
            model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
            encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
        )
        Log.ok("Embedding model loaded")
        return self

    def build(self, domain_chunks: Dict[str, List[Document]]) -> "MultiDomainVectorStore":
        """Index all domain chunks into their respective ChromaDB collections."""
        Log.step("Building ChromaDB collections (one per domain)")

        self._client = chromadb.PersistentClient(
            path=self.cfg.CHROMA_BASE_DIR,
            settings=Settings(anonymized_telemetry=False),
        )

        collection_map = self.cfg.DOMAIN_COLLECTIONS

        for domain_key, chunks in domain_chunks.items():
            coll_name = collection_map.get(domain_key, f"ma_{domain_key}")
            dvs = DomainVectorStore(
                domain_key=domain_key,
                collection_name=coll_name,
                embeddings=self.embeddings,
                cfg=self.cfg,
            )
            dvs.build(chunks, self._client)
            self._domains[domain_key] = dvs

        total = sum(d.doc_count for d in self._domains.values())
        Log.ok(f"Total vectors across {len(self._domains)} collections: {total}")
        return self

    def get_domain(self, domain_key: str) -> Optional[DomainVectorStore]:
        return self._domains.get(domain_key)

    def all_domains(self) -> List[str]:
        return list(self._domains.keys())

    def cross_domain_search(
        self, query: str, k_per_domain: int = 2
    ) -> Dict[str, List[Tuple[Document, float]]]:
        """Search all domains and return per-domain results."""
        results: Dict[str, List[Tuple[Document, float]]] = {}
        for domain_key, dvs in self._domains.items():
            results[domain_key] = dvs.similarity_search(query, k=k_per_domain)
        return results


In [40]:
from abc import ABC, abstractmethod


In [41]:
# ══════════════════════════════════════════════════════════════════════
# LLM FACTORY
# ══════════════════════════════════════════════════════════════════════

def build_llm(cfg: Config, temperature: float = 0.0) -> ChatOllama:
    return ChatOllama(
        base_url=getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
        model=getattr(config, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
        temperature=getattr(config, 'LLM_TEMPERATURE', 0.0),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )


In [42]:


# ══════════════════════════════════════════════════════════════════════
# BASE AGENT
# ══════════════════════════════════════════════════════════════════════

class BaseAgent(ABC):
    """
    Abstract base class for all agents.
    Provides: LLM access, structured logging, message bus integration.
    """

    def __init__(
        self,
        name: str,
        cfg: Config,
        bus: MessageBus,
        llm: Optional[ChatOllama] = None,
    ):
        self.name = name
        self.cfg  = cfg
        self.bus  = bus
        self.llm  = llm
        self._parser = StrOutputParser()

        bus.register(name)
        Log.agent(name, f"registered on message bus")

    def _invoke_llm(self, prompt: ChatPromptTemplate, **kwargs) -> str:
        """Run LLM chain with error handling."""
        if self.llm is None:
            return "[LLM not configured — set Azure OpenAI credentials]"
        try:
            chain = prompt | self.llm | self._parser
            return chain.invoke(kwargs)
        except Exception as exc:
            Log.err(f"{self.name} LLM error: {exc}")
            return f"[LLM error: {exc}]"

    def send(self, receiver: str, msg_type: MsgType, payload: Dict[str, Any]):
        msg = Message(
            sender=self.name,
            receiver=receiver,
            msg_type=msg_type,
            payload=payload,
        )
        self.bus.send(msg)

    def receive(self, timeout: float = 30.0) -> Optional[Message]:
        return self.bus.receive(self.name, timeout=timeout)

    def receive_all(self) -> List[Message]:
        return self.bus.receive_all_waiting(self.name)

    @abstractmethod
    def run(self, *args, **kwargs) -> Any:
        """Main agent logic."""


In [43]:

# ══════════════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════════════

ORCHESTRATOR_ROUTE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are the Orchestrator of a Multi-Agent RAG system.
You have three domain agents:
  1. NLPResearchAgent   — NLP papers: Transformers, BERT, RAG, Self-RAG, language models
  2. SystemsAgent       — Infrastructure: ChromaDB, FAISS, LangChain, AutoGen, vector DBs
  3. MathReasoningAgent — Reasoning methods: CoT, ReAct, FLARE, mathematical proofs, logic

Analyze the user query and output ONLY a valid JSON object (no markdown, no extra text):
{{
  "domains": ["domain1", "domain2"],
  "sub_tasks": {{
    "NLPResearchAgent": "specific sub-task or null",
    "SystemsAgent": "specific sub-task or null",
    "MathReasoningAgent": "specific sub-task or null"
  }},
  "reasoning": "brief explanation of routing decision",
  "query_type": "factual | explanatory | comparison | multi_hop | math"
}}
Domain values must be from: ["nlp_research", "systems_infra", "math_reasoning"]
Include only relevant domains (1–3). Sub-tasks should be specific retrieval queries."""),
    ("human", "Query: {query}"),
])

DOMAIN_AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized {domain_label} research assistant in a multi-agent RAG system.

Your task: answer the sub-question using ONLY the retrieved context below.
Rules:
1. Base every claim strictly on the retrieved context.
2. Cite sources as [Source: document title] after each factual statement.
3. Be precise — include numbers, formulas, and specifics from the context.
4. If context is insufficient, state what is missing.
5. Start your answer with a confidence marker: [HIGH/MEDIUM/LOW confidence]
6. Keep your answer focused — this will be merged with other agents' answers.

Retrieved context:
{context}"""),
    ("human", "Sub-question: {sub_task}\n\nProvide your domain-specific answer:"),
])

CRITIC_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are the Critic Agent in a multi-agent RAG system.
Your role: evaluate partial answers from domain agents for accuracy, consistency, and completeness.

Respond ONLY with valid JSON (no markdown fences):
{{
  "overall_score": 1-10,
  "per_agent_scores": {{
    "NLPResearchAgent": {{"score": 1-10, "issues": []}},
    "SystemsAgent": {{"score": 1-10, "issues": []}},
    "MathReasoningAgent": {{"score": 1-10, "issues": []}}
  }},
  "conflicts": ["list any contradictions between agent answers"],
  "missing_info": ["key information absent from all answers"],
  "synthesis_guidance": "instructions for the Synthesizer on how to merge these answers"
}}"""),
    ("human", """Original query: {query}

Agent answers:
{agent_answers}

Evaluate these answers:"""),
])

SYNTHESIZER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are the Synthesizer Agent in a multi-agent RAG system.
Your role: merge partial answers from multiple domain agents into one coherent, well-cited response.

Instructions:
1. Integrate all relevant information from all agent answers.
2. Resolve any conflicts identified by the Critic (prefer answers with higher confidence).
3. Cite sources as [Source: N] using numbered references.
4. Structure your answer clearly: use paragraphs, not bullet points.
5. The answer must be complete, accurate, and directly address the original query.
6. Do NOT mention the multi-agent architecture — write as a unified expert answer.

Critic's guidance: {synthesis_guidance}
Identified conflicts: {conflicts}"""),
    ("human", """Original query: {query}

Domain agent answers to merge:
{agent_answers}

Write the final synthesized answer:"""),
])


In [44]:


# ══════════════════════════════════════════════════════════════════════
# DOMAIN AGENTS  (NLP, Systems, Math)
# ══════════════════════════════════════════════════════════════════════

class DomainAgent(BaseAgent):
    """
    Generic domain specialist agent.
    Retrieves from its own ChromaDB collection, generates a partial answer,
    reports confidence, and sends result back to orchestrator.
    """

    def __init__(
        self,
        name: str,
        domain_key: str,
        domain_label: str,
        cfg: Config,
        bus: MessageBus,
        llm: ChatOllama,
        domain_vs: DomainVectorStore,
    ):
        super().__init__(name, cfg, bus, llm)
        self.domain_key   = domain_key
        self.domain_label = domain_label
        self.domain_vs    = domain_vs

    def _extract_confidence(self, answer: str) -> float:
        """Parse [HIGH/MEDIUM/LOW confidence] marker from answer."""
        upper = answer.upper()
        if "[HIGH" in upper:
            return 0.9
        if "[MEDIUM" in upper:
            return 0.6
        return 0.3

    def run(self, query: str, sub_task: str) -> AgentResult:
        """Retrieve + generate partial answer for this domain."""
        t0 = time.time()
        Log.agent(self.name, f"retrieving for sub-task: '{sub_task[:60]}'")

        try:
            context = self.domain_vs.get_context_str(sub_task or query, k=self.cfg.TOP_K)
            docs    = self.domain_vs.similarity_search(sub_task or query, k=self.cfg.TOP_K)

            answer = self._invoke_llm(
                DOMAIN_AGENT_PROMPT,
                domain_label=self.domain_label,
                context=context,
                sub_task=sub_task or query,
            )

            sources = list({
                doc.metadata.get("source", "")
                for doc, _ in docs
                if doc.metadata.get("source")
            })

            confidence = self._extract_confidence(answer)
            elapsed    = round(time.time() - t0, 2)

            Log.agent(self.name, f"answered ({len(answer)} chars, confidence={confidence:.1f}, {elapsed}s)")

            return AgentResult(
                agent_name=self.name,
                domain=self.domain_key,
                answer=answer,
                sources=sources,
                doc_count=len(docs),
                confidence=confidence,
                elapsed_sec=elapsed,
            )

        except Exception as exc:
            Log.err(f"{self.name}: {exc}")
            return AgentResult(
                agent_name=self.name,
                domain=self.domain_key,
                answer="",
                sources=[],
                doc_count=0,
                confidence=0.0,
                elapsed_sec=round(time.time() - t0, 2),
                error=str(exc),
            )



In [45]:

# ══════════════════════════════════════════════════════════════════════
# CRITIC AGENT
# ══════════════════════════════════════════════════════════════════════

class CriticAgent(BaseAgent):
    """
    Evaluates partial answers from domain agents.
    Scores accuracy, spots conflicts, and guides the Synthesizer.
    """

    def __init__(self, cfg: Config, bus: MessageBus, llm: ChatOllama):
        super().__init__("CriticAgent", cfg, bus, llm)

    def run(
        self,
        query: str,
        agent_results: List[AgentResult],
    ) -> Dict[str, Any]:
        """Evaluate all agent answers, return structured critique."""
        Log.agent(self.name, f"evaluating {len(agent_results)} agent answers")

        # Format agent answers for the prompt
        formatted = "\n\n".join(
            f"--- {r.agent_name} (domain={r.domain}, confidence={r.confidence:.1f}) ---\n"
            f"{r.answer if r.success else '[AGENT FAILED: ' + str(r.error) + ']'}"
            for r in agent_results
        )

        raw = self._invoke_llm(
            CRITIC_PROMPT,
            query=query,
            agent_answers=formatted,
        )

        try:
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            critique = json.loads(clean)
        except json.JSONDecodeError:
            Log.warn("Critic JSON parse failed — using default critique")
            critique = {
                "overall_score": 7,
                "per_agent_scores": {},
                "conflicts": [],
                "missing_info": [],
                "synthesis_guidance": "Merge all agent answers, prioritise high-confidence sections.",
            }

        score = critique.get("overall_score", 7)
        conflicts = critique.get("conflicts", [])
        Log.agent(self.name, f"score={score}/10, conflicts={len(conflicts)}")
        if conflicts:
            for c in conflicts:
                Log.warn(f"  Conflict: {c}")

        return critique



In [46]:

# ══════════════════════════════════════════════════════════════════════
# SYNTHESIZER AGENT
# ══════════════════════════════════════════════════════════════════════

class SynthesizerAgent(BaseAgent):
    """
    Merges all domain agent answers (guided by Critic) into one
    coherent, well-cited final answer.
    """

    def __init__(self, cfg: Config, bus: MessageBus, llm: ChatOllama):
        super().__init__("SynthesizerAgent", cfg, bus, llm)

    def run(
        self,
        query: str,
        agent_results: List[AgentResult],
        critique: Dict[str, Any],
    ) -> str:
        """Synthesize a final answer from all partial answers."""
        Log.agent(self.name, "synthesising final answer")

        # Only include successful, non-empty answers
        active_results = [r for r in agent_results if r.success]
        if not active_results:
            return "No domain agents produced usable results."

        formatted = "\n\n".join(
            f"[{r.agent_name} — {r.domain}]\n{r.answer}"
            for r in active_results
        )

        synthesis_guidance = critique.get(
            "synthesis_guidance",
            "Merge all agent answers into a coherent response.",
        )
        conflicts = json.dumps(critique.get("conflicts", []))

        answer = self._invoke_llm(
            SYNTHESIZER_PROMPT,
            query=query,
            agent_answers=formatted,
            synthesis_guidance=synthesis_guidance,
            conflicts=conflicts,
        )

        Log.agent(self.name, f"final answer: {len(answer)} chars")
        return answer



In [47]:


# ══════════════════════════════════════════════════════════════════════
# ORCHESTRATOR AGENT
# ══════════════════════════════════════════════════════════════════════

class OrchestratorAgent(BaseAgent):
    """
    Central coordinator for the multi-agent RAG system.

    Responsibilities:
      1. Analyze the query and route to relevant domain agents
      2. Dispatch domain agents (parallel or sequential)
      3. Collect partial answers via message bus
      4. Request critique from CriticAgent
      5. Request final synthesis from SynthesizerAgent
      6. Return structured result
    """

    def __init__(
        self,
        cfg: Config,
        bus: MessageBus,
        llm: ChatOllama,
        mvs: MultiDomainVectorStore,
    ):
        super().__init__("Orchestrator", cfg, bus, llm)
        self.mvs = mvs

        # Build domain agents
        self._domain_agents: Dict[str, DomainAgent] = self._build_domain_agents(llm)
        self._critic     = CriticAgent(cfg, bus, llm)
        self._synthesizer = SynthesizerAgent(cfg, bus, llm)

        Log.agent(self.name, (
            f"online — {len(self._domain_agents)} domain agents + "
            "CriticAgent + SynthesizerAgent"
        ))

    def _build_domain_agents(self, llm: ChatOllama) -> Dict[str, DomainAgent]:
        agents = {}
        specs = [
            ("NLPResearchAgent",   "nlp_research",   "NLP Research"),
            ("SystemsAgent",       "systems_infra",  "Systems & Infrastructure"),
            ("MathReasoningAgent", "math_reasoning", "Math & Reasoning"),
        ]
        for agent_name, domain_key, label in specs:
            dvs = self.mvs.get_domain(domain_key)
            if dvs is None:
                Log.warn(f"Domain '{domain_key}' not found in vector store")
                continue
            agents[agent_name] = DomainAgent(
                name=agent_name,
                domain_key=domain_key,
                domain_label=label,
                cfg=self.cfg,
                bus=self.bus,
                llm=llm,
                domain_vs=dvs,
            )
        return agents

    def _route_query(self, query: str) -> Dict[str, Any]:
        """Ask LLM to classify query and assign sub-tasks per domain."""
        Log.agent(self.name, "routing query to domain agents")
        raw = self._invoke_llm(ORCHESTRATOR_ROUTE_PROMPT, query=query)
        try:
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            routing = json.loads(clean)
        except json.JSONDecodeError:
            Log.warn("Route JSON parse failed — defaulting to all agents")
            routing = {
                "domains":    ["nlp_research", "systems_infra", "math_reasoning"],
                "sub_tasks":  {
                    "NLPResearchAgent":   query,
                    "SystemsAgent":       query,
                    "MathReasoningAgent": query,
                },
                "reasoning":  "fallback: all agents",
                "query_type": "factual",
            }

        Log.agent(self.name, (
            f"type={routing.get('query_type')} | "
            f"domains={routing.get('domains')} | "
            f"reason={routing.get('reasoning','')[:60]}"
        ))
        return routing

    def _run_domain_agent(
        self,
        agent_name: str,
        query: str,
        sub_task: str,
        results_list: List[AgentResult],
        lock: threading.Lock,
    ):
        """Thread target: run one domain agent and store result."""
        agent = self._domain_agents.get(agent_name)
        if agent is None:
            return
        result = agent.run(query=query, sub_task=sub_task)
        with lock:
            results_list.append(result)

    def run(self, query: str) -> Dict[str, Any]:
        """
        Full multi-agent pipeline:
          Route → Dispatch (parallel) → Collect → Critique → Synthesize → Return
        """
        t0 = time.time()
        Log.section(f"MULTI-AGENT QUERY: {query}")

        # ── Step 1: Route ──────────────────────────────────────────────
        routing   = self._route_query(query)
        sub_tasks = routing.get("sub_tasks", {})

        # Determine which agents to activate
        active_agents = [
            a for a in self._domain_agents
            if sub_tasks.get(a) is not None
        ]
        if not active_agents:
            active_agents = list(self._domain_agents.keys())

        Log.agent(self.name, f"activating {len(active_agents)} domain agents: {active_agents}")

        # ── Step 2: Dispatch domain agents ────────────────────────────
        agent_results: List[AgentResult] = []

        if self.cfg.ENABLE_PARALLEL and len(active_agents) > 1:
            Log.agent(self.name, "running domain agents IN PARALLEL")
            lock    = threading.Lock()
            threads = []
            for agent_name in active_agents:
                sub_task = sub_tasks.get(agent_name) or query
                t = threading.Thread(
                    target=self._run_domain_agent,
                    args=(agent_name, query, sub_task, agent_results, lock),
                    daemon=True,
                )
                threads.append(t)
                t.start()
            for t in threads:
                t.join(timeout=self.cfg.AGENT_TIMEOUT_SEC)
        else:
            Log.agent(self.name, "running domain agents SEQUENTIALLY")
            for agent_name in active_agents:
                sub_task = sub_tasks.get(agent_name) or query
                result   = self._domain_agents[agent_name].run(
                    query=query, sub_task=sub_task
                )
                agent_results.append(result)

        successful = [r for r in agent_results if r.success]
        Log.agent(
            self.name,
            f"collected {len(successful)}/{len(agent_results)} successful agent results",
        )

        # ── Step 3: Critique ───────────────────────────────────────────
        critique = self._critic.run(query, agent_results)

        # ── Step 4: Synthesize ─────────────────────────────────────────
        final_answer = self._synthesizer.run(query, agent_results, critique)

        # ── Step 5: Assemble result metadata ──────────────────────────
        all_sources: set[str] = set()
        for r in agent_results:
            for src in r.sources:
                if src:
                    all_sources.add(src)

        elapsed = round(time.time() - t0, 2)
        result: Dict[str, Any] = {
            "query":          query,
            "answer":         final_answer,
            "routing":        routing,
            "agent_results":  agent_results,
            "critique":       critique,
            "critique_score": critique.get("overall_score"),
            "agents_used":    [r.agent_name for r in agent_results],
            "total_docs":     sum(r.doc_count for r in agent_results),
            "sources":        list(all_sources),
            "message_count":  self.bus.message_count,
            "elapsed":        elapsed,
        }

        Log.answer_box(query, final_answer, result)
        return result


In [48]:

# ══════════════════════════════════════════════════════════════════════
# DEMO QUERIES  — 8 queries exercising all routing combinations
# ══════════════════════════════════════════════════════════════════════

DEMO_QUERIES = [
    # 1. NLP-only
    (
        "What is Multi-Head Attention and how does it differ from single-head attention in the Transformer?",
        "Agents: NLPResearchAgent"
    ),
    # 2. Systems-only
    (
        "How does ChromaDB's MMR retrieval algorithm balance relevance and diversity, and what is its mathematical formula?",
        "Agents: SystemsAgent"
    ),
    # 3. Math/Reasoning-only
    (
        "How does Chain-of-Thought prompting improve LLM performance on GSM8K, and what accuracy does PaLM 540B achieve?",
        "Agents: MathReasoningAgent"
    ),
    # 4. NLP + Systems (comparison)
    (
        "Compare RAG (Lewis et al.) and Self-RAG in terms of retrieval strategy, and explain how LangChain implements RAG chains.",
        "Agents: NLPResearchAgent + SystemsAgent"
    ),
    # 5. NLP + Math (multi-hop)
    (
        "How does BERT's pre-training compare to FLARE's active retrieval strategy for handling uncertainty in generated text?",
        "Agents: NLPResearchAgent + MathReasoningAgent"
    ),
    # 6. Systems + Math
    (
        "How do AutoGen multi-agent systems improve on single-agent performance on MATH benchmark, and how does ReAct's reasoning loop help reduce hallucinations?",
        "Agents: SystemsAgent + MathReasoningAgent"
    ),
    # 7. All three agents (broad)
    (
        "Describe a complete end-to-end RAG pipeline: from embedding documents in ChromaDB, to retrieving with MMR, to generating with a Transformer model, and validating with chain-of-thought reasoning.",
        "Agents: NLPResearchAgent + SystemsAgent + MathReasoningAgent"
    ),
    # 8. Specific factual + math
    (
        "What BLEU score did the original Transformer achieve on English-to-German translation, and how does that compare to BERT's SQuAD F1 score?",
        "Agents: NLPResearchAgent + MathReasoningAgent"
    ),
]



In [49]:

# ══════════════════════════════════════════════════════════════════════
# SYSTEM BUILDER
# ══════════════════════════════════════════════════════════════════════

def build_system(cfg: Config) -> OrchestratorAgent:
    """Full initialization pipeline."""
    Log.banner("MULTI-AGENT RETRIEVAL RAG — SYSTEM INITIALIZATION")

    # 1. Documents
    Log.step("Processing domain corpus")
    processor    = DocumentProcessor(cfg)
    domain_chunks = processor.build_domain_chunks()

    # 2. Multi-domain vector store
    mvs = MultiDomainVectorStore(cfg)
    mvs.init_embeddings()
    mvs.build(domain_chunks)

    # 3. Message bus
    bus = MessageBus()

    # 4. LLM
    if cfg.is_configured():
        Log.step("Connecting to Azure OpenAI")
        llm = build_llm(cfg)
        Log.ok(f"LLM ready — deployment: {cfg.OLLAMA_MODEL}")
    else:
        Log.warn("Azure OpenAI credentials not set — LLM calls will fail gracefully")
        Log.warn("Set: OLLAMA_BASE_URL, OLLAMA_BASE_URL, OLLAMA_MODEL")
        llm = build_llm(cfg)  # will fail at call time, not at construction

    # 5. Orchestrator (builds all sub-agents internally)
    orchestrator = OrchestratorAgent(cfg=cfg, bus=bus, llm=llm, mvs=mvs)

    Log.banner("SYSTEM READY")

    # Print reference table
    Log.step("Reference datasets / papers:")
    for i, (title, url) in enumerate(ALL_REFERENCES, 1):
        Log.info(f"  [{i:2d}] {title}")
        Log.info(f"        {url}")

    return orchestrator



In [25]:

# ══════════════════════════════════════════════════════════════════════
# RUN MODES
# ══════════════════════════════════════════════════════════════════════

def run_demo(orch: OrchestratorAgent):
    """Run all 8 pre-set demo queries."""
    print(f"\n{C.BOLD}{C.CYAN}Running {len(DEMO_QUERIES)} demo queries...{C.RESET}")
    for i, (query, note) in enumerate(DEMO_QUERIES, 1):
        print(f"\n{C.BOLD}{C.YELLOW}━━ Demo {i}/{len(DEMO_QUERIES)}: {note} ━━{C.RESET}")
        try:
            orch.run(query)
        except Exception as exc:
            Log.err(f"Demo {i} failed: {exc}")
        time.sleep(0.5)


def run_single_demo(orch: OrchestratorAgent, n: int):
    """Run a specific demo query by 1-based index."""
    idx = n - 1
    if not (0 <= idx < len(DEMO_QUERIES)):
        Log.err(f"demo-n must be 1–{len(DEMO_QUERIES)}")
        sys.exit(1)
    query, note = DEMO_QUERIES[idx]
    print(f"\n{C.DIM}Demo {n}: {note}{C.RESET}")
    orch.run(query)


def run_interactive(orch: OrchestratorAgent):
    """Interactive multi-turn CLI."""
    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║          MULTI-AGENT RETRIEVAL RAG  —  Interactive CLI          ║")
    print("║  Commands:  'demo N'  |  'agents'  |  'refs'  |  'quit'         ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    print(C.RESET)
    print(f"{C.DIM}  Agents: Orchestrator · NLPResearchAgent · SystemsAgent · MathReasoningAgent")
    print(f"         CriticAgent · SynthesizerAgent{C.RESET}\n")

    while True:
        try:
            user = input(f"\n{C.BOLD}You:{C.RESET} ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user:
            continue

        cmd = user.lower()

        if cmd in ("quit", "exit", "q"):
            print("Goodbye!")
            break

        if cmd == "agents":
            print(f"\n{C.MAGENTA}Active agents:{C.RESET}")
            for name in ["Orchestrator", "NLPResearchAgent", "SystemsAgent",
                         "MathReasoningAgent", "CriticAgent", "SynthesizerAgent"]:
                print(f"  • {name}")
            continue

        if cmd == "refs":
            for i, (title, url) in enumerate(ALL_REFERENCES, 1):
                print(f"  [{i}] {title} — {url}")
            continue

        if cmd.startswith("demo"):
            parts = cmd.split()
            if len(parts) > 1 and parts[1].isdigit():
                run_single_demo(orch, int(parts[1]))
            else:
                print(f"  Demo queries (1–{len(DEMO_QUERIES)}):")
                for i, (q, note) in enumerate(DEMO_QUERIES, 1):
                    print(f"  {i}. [{note}] {q[:70]}...")
            continue

        try:
            orch.run(user)
        except Exception as exc:
            Log.err(f"Error: {exc}")
